# Crypto Narrative & NLP: offline research pipeline

Run all cells from repository root or notebooks/. Inputs: data/processed/market_daily.parquet and text_daily.parquet. No network calls. If real text is absent, the notebook explicitly uses seeded synthetic fixtures and labels every artifact accordingly. Never mix real prices with synthetic text.

Required environment: project requirements plus matplotlib, nbformat, nbclient and ipykernel. Existing project .venv packages can be reused by the host Jupyter kernel.


In [1]:
from pathlib import Path
import sys, json, hashlib, subprocess
import pandas as pd
import yaml
root = Path.cwd()
if root.name == 'notebooks':
    root = root.parent
assert (root / 'config/config.yaml').is_file(), 'Run from repository root or notebooks/'
sys.path.insert(0, str(root))
# Reuse installed project research packages with the host notebook kernel.
project_packages = root / '.venv' / ('Lib/site-packages' if sys.platform == 'win32' else f'lib/python{sys.version_info.major}.{sys.version_info.minor}/site-packages')
if project_packages.exists():
    sys.path.insert(1, str(project_packages))
from src.models.run import run_analysis
config = yaml.safe_load((root/'config/config.yaml').read_text(encoding='utf-8'))
real = (root/'data/processed/text_daily.parquet').is_file()
if real:
    market = pd.read_parquet(root/'data/processed/market_daily.parquet')
    text = pd.read_parquet(root/'data/processed/text_daily.parquet')
else:
    import importlib.util
    spec = importlib.util.spec_from_file_location('fixture_data', root/'tests/fixtures/make_fixtures.py')
    fixture_data = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(fixture_data)
    market, text = fixture_data.make_fixtures(config)
mode = 'real' if real else 'synthetic'
print(mode.upper(), 'market rows', len(market), 'text rows', len(text))
ucry_path = root/'data/processed/ucry_daily.parquet'
ucry = pd.read_parquet(ucry_path) if real and ucry_path.exists() else None

from src.evaluation.hourly_event import hourly_prices
interim = root/'data/interim'
event_inputs = (pd.read_parquet(interim/'documents.parquet'), pd.read_parquet(interim/'doc_scores.parquet'), hourly_prices()) if real and (interim/'documents.parquet').exists() and (interim/'doc_scores.parquet').exists() else None


REAL market rows 6845 text rows 7326


## Recompute all tables and paper figures

Core spec uses only same-day text; market return lags remain available on sparse news dates. Real runs also read cached hourly Binance bars and canonical articles for event tests. Tercile cutoffs use training articles only. Every refit purges horizon plus embargo; no network calls.


In [2]:
output = root/'outputs/tables'/mode
figures = root/'outputs/figures'/mode
tables = run_analysis(market, text, config, output, figures, real=real, ucry=ucry, event_inputs=event_inputs)
print('Written:', output, figures)
inputs = [root/'data/processed/market_daily.parquet', root/'data/processed/text_daily.parquet']
if ucry is not None:
    inputs.append(ucry_path)
if event_inputs is not None:
    inputs += [interim/'documents.parquet', interim/'doc_scores.parquet']
    inputs += sorted((root/'data/raw').glob('*_1h_*.parquet'))
code_paths = [root/'src/models/run.py', root/'src/models/ml.py', root/'src/evaluation/report.py', root/'src/evaluation/hourly_event.py']
metadata = {'result_source': mode, 'config': config, 'refit_days': 30,
            'pipeline_commit': subprocess.check_output(['git','rev-parse','HEAD'], cwd=root, text=True).strip(),
            'sha256': {str(path): hashlib.sha256(path.read_bytes()).hexdigest() for path in inputs},
            'analysis_code_sha256': hashlib.sha256(b''.join(path.read_bytes() for path in code_paths)).hexdigest()}
(output/'run_metadata.json').write_text(json.dumps(metadata, indent=2)+'\n', encoding='utf-8')


Running econometrics


Running robustness


Running quantiles


Running expanding forecasts


ML: BNB, h=1, target=ret


...

ML: XRP, h=7, target=ret


ML: XRP, h=7, target=rv


Running hourly events


Written: C:\Users\Dell\crypto-narrative-nlp\outputs\tables\real C:\Users\Dell\crypto-narrative-nlp\outputs\figures\real


3626

In [3]:
display(tables['coverage'])
for name in ['regressions','wald','quantiles','granger','oos','gw','mcs','ml_status']:
    print(name, len(tables[name]), 'rows')
    display(tables[name].head(10))
audit = tables['refit_audit']
if not audit.empty:
    assert (audit.train_target_max + pd.to_timedelta(audit.horizon+audit.embargo, unit='D') < audit.origin).all()
print((output/'README.md').read_text(encoding='utf-8'))

if real:
    for name in ['event_coverage', 'event_tests', 'event_asymmetry', 'event_timing']:
        print(name, len(tables.get(name, [])), 'rows')
        display(tables.get(name, pd.DataFrame()).head(10))
    print((output/'RESULTS.md').read_text(encoding='utf-8'))


,asset,calendar_days,news_days,pct_days_with_news,asset_news_days,pct_days_with_asset_news,usable_model_days,fallback_model_days,finbert_days,pct_days_with_finbert,model_development_days,model_test_days,model_train_days,model_valid_days,news_start,news_end,model_start,model_end,configured_start,configured_end
0,BNB,1369,1352,98.758218,801,58.509861,866,551,1352,98.758218,677,189,556,121,2023-01-01 00:00:00+00:00,2026-09-30 00:00:00+00:00,2023-01-01 00:00:00+00:00,2026-09-29 00:00:00+00:00,2023-01-01,2026-09-30
1,BTC,1369,1352,98.758218,1351,98.685172,1344,1,1352,98.758218,1072,272,890,182,2023-01-01 00:00:00+00:00,2026-09-30 00:00:00+00:00,2023-01-01 00:00:00+00:00,2026-09-29 00:00:00+00:00,2023-01-01,2026-09-30
2,ETH,1369,1352,98.758218,1321,96.493791,1253,31,1352,98.758218,1011,242,838,173,2023-01-01 00:00:00+00:00,2026-09-30 00:00:00+00:00,2023-01-01 00:00:00+00:00,2026-09-29 00:00:00+00:00,2023-01-01,2026-09-30
3,SOL,1369,1352,98.758218,1212,88.531775,1035,140,1352,98.758218,864,171,716,148,2023-01-01 00:00:00+00:00,2026-09-30 00:00:00+00:00,2023-01-01 00:00:00+00:00,2026-09-29 00:00:00+00:00,2023-01-01,2026-09-30
4,XRP,1369,1352,98.758218,1272,92.914536,1179,79,1352,98.758218,961,218,784,177,2023-01-01 00:00:00+00:00,2026-09-30 00:00:00+00:00,2023-01-01 00:00:00+00:00,2026-09-29 00:00:00+00:00,2023-01-01,2026-09-30


regressions 4950 rows


,asset,horizon,target,sentiment,version,term,coef,se,pvalue,identified,nobs,resid_df,r2,sample_start,sample_end,target_max,p_holm,p_bh
0,BNB,1,ret,none,market,const,0.073651,0.041962,0.079228,True,677.0,666.0,0.026424,2023-01-01 00:00:00+00:00,2025-12-30 00:00:00+00:00,2025-12-31 00:00:00+00:00,1.0,0.377942
1,BNB,1,ret,none,market,ret,0.023607,0.042485,0.578449,True,677.0,666.0,0.026424,2023-01-01 00:00:00+00:00,2025-12-30 00:00:00+00:00,2025-12-31 00:00:00+00:00,1.0,0.837389
2,BNB,1,ret,none,market,rv,2.211026,1.565042,0.157727,True,677.0,666.0,0.026424,2023-01-01 00:00:00+00:00,2025-12-30 00:00:00+00:00,2025-12-31 00:00:00+00:00,1.0,0.519460
3,BNB,1,ret,none,market,logvol,-0.003967,0.002276,0.081289,True,677.0,666.0,0.026424,2023-01-01 00:00:00+00:00,2025-12-30 00:00:00+00:00,2025-12-31 00:00:00+00:00,1.0,0.383583
4,BNB,1,ret,none,market,ret_lag1,0.012666,0.041987,0.762913,True,677.0,666.0,0.026424,2023-01-01 00:00:00+00:00,2025-12-30 00:00:00+00:00,2025-12-31 00:00:00+00:00,1.0,0.933140
5,BNB,1,ret,none,market,ret_lag2,0.065587,0.055525,0.237511,True,677.0,666.0,0.026424,2023-01-01 00:00:00+00:00,2025-12-30 00:00:00+00:00,2025-12-31 00:00:00+00:00,1.0,0.591984
6,BNB,1,ret,none,market,ret_lag3,0.053078,0.037968,0.162124,True,677.0,666.0,0.026424,2023-01-01 00:00:00+00:00,2025-12-30 00:00:00+00:00,2025-12-31 00:00:00+00:00,1.0,0.521790
7,BNB,1,ret,none,market,ret_lag4,0.010483,0.039846,0.792484,True,677.0,666.0,0.026424,2023-01-01 00:00:00+00:00,2025-12-30 00:00:00+00:00,2025-12-31 00:00:00+00:00,1.0,0.939593
8,BNB,1,ret,none,market,ret_lag5,-0.022777,0.052281,0.663077,True,677.0,666.0,0.026424,2023-01-01 00:00:00+00:00,2025-12-30 00:00:00+00:00,2025-12-31 00:00:00+00:00,1.0,0.888276
9,BNB,1,ret,none,market,ret_lag6,0.004018,0.046508,0.931151,True,677.0,666.0,0.026424,2023-01-01 00:00:00+00:00,2025-12-30 00:00:00+00:00,2025-12-31 00:00:00+00:00,1.0,0.980264


wald 15 rows


,asset,horizon,beta_neg,beta_pos,identified,nobs,resid_df,sample_start,sample_end,statistic,pvalue,p_holm,p_bh
0,BNB,1,0.000066,-0.000041,True,677.0,654.0,2023-01-01 00:00:00+00:00,2025-12-30 00:00:00+00:00,0.087437,0.767461,1.000000,0.972125
1,BTC,1,-0.001256,-0.001084,True,1072.0,1049.0,2023-01-01 00:00:00+00:00,2025-12-30 00:00:00+00:00,0.152152,0.696488,1.000000,0.972125
2,ETH,1,0.000656,0.001390,True,1011.0,988.0,2023-01-01 00:00:00+00:00,2025-12-30 00:00:00+00:00,3.048916,0.080791,1.000000,0.403955
3,SOL,1,0.000640,0.000039,True,864.0,841.0,2023-01-01 00:00:00+00:00,2025-12-30 00:00:00+00:00,0.521842,0.470057,1.000000,0.903732
4,XRP,1,0.002330,-0.000829,True,961.0,938.0,2023-01-01 00:00:00+00:00,2025-12-30 00:00:00+00:00,0.957197,0.327894,1.000000,0.825892
5,BNB,3,-0.000227,-0.000279,True,675.0,652.0,2023-01-01 00:00:00+00:00,2025-12-28 00:00:00+00:00,0.033176,0.855470,1.000000,0.972125
6,BTC,3,0.000534,0.000550,True,1070.0,1047.0,2023-01-01 00:00:00+00:00,2025-12-28 00:00:00+00:00,0.001221,0.972125,1.000000,0.972125
7,ETH,3,-0.001081,-0.000155,True,1009.0,986.0,2023-01-01 00:00:00+00:00,2025-12-28 00:00:00+00:00,3.266904,0.070691,0.989673,0.403955
8,SOL,3,0.004081,0.001022,True,861.0,838.0,2023-01-01 00:00:00+00:00,2025-12-28 00:00:00+00:00,6.497635,0.010802,0.162027,0.162027
9,XRP,3,-0.001920,-0.001989,True,959.0,936.0,2023-01-01 00:00:00+00:00,2025-12-28 00:00:00+00:00,0.008362,0.927141,1.000000,0.972125


...

event_asymmetry 24 rows


,tone,window,metric,n_days,negative_mean,positive_mean,negative_minus_positive,p_t,p_wilcoxon,p_t_holm,p_wilcoxon_holm
0,finbert_tercile,"[0,1]",car,1352,0.000039,-0.000081,0.000121,2.817345e-02,4.426128e-03,1.972141e-01,5.753967e-02
1,finbert_tercile,"[0,1]",absret,1352,0.000298,0.000459,-0.000161,1.008291e-03,1.136674e-02,1.310778e-02,1.364009e-01
2,finbert_tercile,"[0,1]",volume,1352,0.041943,0.072700,-0.030757,9.529383e-11,1.987734e-13,2.001170e-09,4.373015e-12
3,finbert_tercile,"[0,4]",car,1352,-0.000020,-0.000159,0.000139,9.335654e-02,4.774451e-02,4.307645e-01,3.450627e-01
4,finbert_tercile,"[0,4]",absret,1352,0.000474,0.000714,-0.000239,4.921497e-03,5.743804e-02,5.413647e-02,3.450627e-01
5,finbert_tercile,"[0,4]",volume,1352,0.022285,0.050545,-0.028259,4.155174e-14,1.171517e-15,9.972419e-13,2.811640e-14
6,finbert_tercile,"[0,24]",car,1351,-0.000153,-0.000637,0.000485,4.102490e-03,1.885988e-02,4.922987e-02,2.074587e-01
7,finbert_tercile,"[0,24]",absret,1351,0.000408,0.000729,-0.000321,1.495658e-01,8.889899e-01,4.307645e-01,1.000000e+00
8,finbert_tercile,"[0,24]",volume,1351,0.002555,0.011956,-0.009401,5.184523e-06,1.082941e-04,8.813688e-05,1.841000e-03
9,finbert_tercile,"[-4,0]",car,1352,0.000182,-0.000225,0.000407,5.499725e-07,1.142594e-06,1.044948e-05,2.056669e-05


event_timing 3 rows


,window,n_pairs,n_days,n_unique_articles,mean_car_published,mean_car_seen,mean_published_minus_seen,p_paired_t,p_paired_wilcoxon,p_public,p_seen,significance_changed,p_paired_t_holm,p_paired_wilcoxon_holm,p_public_holm,p_seen_holm,significance_changed_holm
0,0_1,38,31,36,-0.000078,-0.001269,0.001191,0.362440,0.542198,0.950745,0.099590,False,0.724881,1.00000,0.950745,0.199181,False
1,0_4,38,31,36,-0.002471,-0.001988,-0.000483,0.820783,0.915394,0.143801,0.221002,False,0.820783,1.00000,0.431404,0.221002,False
2,0_24,38,31,36,-0.005025,-0.008663,0.003638,0.120842,0.163340,0.204898,0.025924,True,0.362525,0.49002,0.431404,0.077773,False


# Kết quả thật, bộ tin đang thu

Bằng chứng dưới đây chỉ áp dụng cho dữ liệu đã thu. Tin cùng ngày được gom theo ngày khi kiểm định sự kiện.

Coverage: corpus chung có 1352/1369 ngày có tin (98.76%); riêng BTC có 1351 ngày tin gắn mã (98.69%), 1 ngày mô hình dùng corpus chung; từ 2023-01-01 00:00:00+00:00 đến 2026-09-30 00:00:00+00:00. Core spec dùng 1344 ngày BTC, train/valid/test=890/182/272.

Sự kiện giờ: 142646 tin canonical tiếng Anh, 150160 cặp tin–asset trên 1363 ngày; 140593 tin có FinBERT. Mẫu 2023-01-01 02:00:00+00:00 đến 2026-10-01 00:00:00+00:00.

Bản documents hiện phủ 1352 ngày GKG và 503 ngày SEC; text_daily có thể chưa được build lại trên cùng ảnh chụp dữ liệu.

Phản ứng chung [0,4]h car: -0.006788 %, 1363 ngày, p t Holm=1.
Phản ứng chung [0,4]h absret: 0.05448 %, 1363 ngày, p t Holm=1.
Phản ứng chung [0,4]h volume: 0.03311 log điểm, 1363 ngày, p t Holm=1.
Abnormal volume là chênh log(1+USD volume) so với 144 giờ ước lượng trước tin; abnormal |return| là chênh tổng độ l

## Interpretation

Read `outputs/tables/real/RESULTS.md` for RQ1?RQ5 and `README.md` for coverage and each table. Hourly events cluster by day. The 24-hour placebo is omitted where a ?24h shift would overlap T0. Synthetic findings validate code only; strategy plots are economic-value tests.
